# 04 Deduplication of multiple listings

Without the Major and Primary filters the universe can contain more than one line per company. Lines with the same name in the same country are clustered: two lines are merged only if their quotation periods overlap by at least a quarter of the shorter one and they report the same ICB sector. One line is kept per cluster, the one with the longest history, with average market value as tie-break. Re-listings and distinct firms sharing a name are kept. Section 3.4.3.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

monthly = pd.read_parquet(os.path.join(DATA_DIR, "clean_monthly.parquet"))
weekly  = pd.read_parquet(os.path.join(DATA_DIR, "clean_weekly.parquet"))
trunc   = pd.read_parquet(os.path.join(DATA_DIR, "truncation_points.parquet"))
master  = pd.read_csv(os.path.join(DATA_DIR, "master_universe.csv"), dtype=str)

trunc = trunc.set_index("symbol") if "symbol" in trunc.columns else trunc.set_index(trunc.columns[0])
print(f"monthly {len(monthly):>12,} rows | {monthly['symbol'].nunique():>6,} securities")
print(f"weekly  {len(weekly):>12,} rows | {weekly['symbol'].nunique():>6,} securities")
print(f"universe{len(master):>12,} rows")

## Candidate groups

In [ ]:
alive = set(monthly["symbol"])
u = master[master["Symbol"].isin(alive)].copy()
print(f"Securities with usable monthly data: {len(u):,}")

grp_key = ["country", "full_name"]
sizes = u.groupby(grp_key)["Symbol"].size()
dup_keys = sizes[sizes > 1]

cand = u.merge(dup_keys.rename("n_lines"), left_on=grp_key, right_index=True)
print(f"\nGroups with more than one line: {len(dup_keys):,}")
print(f"Securities involved            : {len(cand):,} "
      f"({100*len(cand)/len(u):.1f}% of the sample)")
print("\nGroup size distribution:")
print(dup_keys.value_counts().sort_index().rename("groups").to_frame().T.to_string())

## Listing periods

In [ ]:
cand = cand.join(trunc[["first_obs", "last_real"]], on="Symbol")

n_missing = int(cand["first_obs"].isna().sum())
print(f"Candidates without a period (no usable RI): {n_missing:,}")
if n_missing:
    print("  These are excluded from clustering and retained unchanged.")
cand = cand[cand["first_obs"].notna()].copy()

# average market capitalisation, used as tie-break
mv = (monthly[monthly["datatype"] == "MV"]
      .groupby("symbol")["value"].mean().rename("avg_mv"))
cand = cand.join(mv, on="Symbol")
cand["n_months"] = ((cand["last_real"] - cand["first_obs"]).dt.days / 30.44).round()

print(f"\nCandidates with a measurable period: {len(cand):,}")
print(cand[["n_months"]].describe().round(0).to_string())

## Clustering by period and sector

In [ ]:
MIN_OVERLAP = 0.25


def overlap_clusters(g, min_overlap=MIN_OVERLAP):
    """
    Cluster the lines of a name group into listing episodes.

    Two lines are merged only if their quotation periods overlap by at least
    `min_overlap` of the shorter period AND they report the same ICB sector.
    A differing sector indicates distinct businesses sharing a name rather than
    duplicate listings of one company; where the sector is unrecorded the lines
    are left unmerged, which errs towards retaining data.
    """
    g = g.reset_index(drop=True)
    n = len(g)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            si, sj = g.loc[i, "Sector"], g.loc[j, "Sector"]
            if pd.isna(si) or pd.isna(sj) or si != sj:
                continue                      # distinct entities, or unknown

            a1, b1 = g.loc[i, "first_obs"], g.loc[i, "last_real"]
            a2, b2 = g.loc[j, "first_obs"], g.loc[j, "last_real"]
            lo, hi = max(a1, a2), min(b1, b2)
            ov = max(0, (hi - lo).days)
            lmin = min((b1 - a1).days, (b2 - a2).days)
            share = ov / lmin if lmin > 0 else (1.0 if ov > 0 else 0.0)
            if share >= min_overlap:
                union(i, j)

    g["cluster"] = [find(i) for i in range(n)]
    return g


parts = []
for _, g in cand.groupby(grp_key):
    parts.append(overlap_clusters(g))
cl = pd.concat(parts, ignore_index=True)
cl["group_id"] = cl["country"] + "|" + cl["full_name"] + "|" + cl["cluster"].astype(str)

n_groups = cl.groupby(grp_key).ngroups
n_clusters = cl["group_id"].nunique()
print(f"Name groups            : {n_groups:,}")
print(f"Listing episodes       : {n_clusters:,}")
print(f"Securities to discard  : {len(cl) - n_clusters:,}")

### Group composition

How groups split between concurrent listings, re-listings, and mixed cases.

In [ ]:
summary = (cl.groupby(grp_key)
           .agg(n_lines=("Symbol", "size"), n_episodes=("cluster", "nunique"),
                n_sectors=("Sector", "nunique")))
summary["type"] = np.where(summary["n_episodes"] == 1, "single episode",
                    np.where(summary["n_episodes"] == summary["n_lines"],
                             "all distinct", "mixed"))
print(summary["type"].value_counts().to_string())

summary["removed"] = summary["n_lines"] - summary["n_episodes"]
print("\nLines removed by group type:")
print(summary.groupby("type")["removed"].sum().to_string())

multi = summary[summary["n_sectors"] > 1]
print(f"\nName groups spanning more than one sector: {len(multi):,}")
print("These are kept as separate entities rather than merged.")

## Selection within each episode

In [ ]:
cl = cl.sort_values(["group_id", "n_months", "avg_mv"],
                    ascending=[True, False, False])
cl["keep"] = ~cl.duplicated(subset="group_id", keep="first")

keep_syms = set(cl.loc[cl["keep"], "Symbol"])
drop_syms = set(cl.loc[~cl["keep"], "Symbol"])

print(f"Retained : {len(keep_syms):,}")
print(f"Discarded: {len(drop_syms):,}")

disc = cl.loc[~cl["keep"]]
print("\nDiscarded by country:")
print(disc["country"].value_counts().to_string())
print("\nDiscarded by status:")
print(disc["Activity"].value_counts().to_string())

### Inspection

Groups in which at least one line is discarded (first 15).

In [ ]:
cols = ["Symbol", "Activity", "Sector", "first_obs", "last_real",
        "n_months", "avg_mv", "cluster", "keep"]
shown = 0
for (co, nm), g in cl.groupby(grp_key):
    if g["keep"].all():
        continue           # nothing discarded here
    print(f"\n{co}, {nm}")
    gg = g[cols].copy()
    gg["first_obs"] = gg["first_obs"].dt.strftime("%Y-%m")
    gg["last_real"] = gg["last_real"].dt.strftime("%Y-%m")
    gg["avg_mv"] = gg["avg_mv"].round(0)
    gg["Sector"] = gg["Sector"].str[:28]
    print(gg.to_string(index=False))
    shown += 1
    if shown >= 15:
        print(f"\n... ({(~cl['keep']).groupby([cl['country'], cl['full_name']]).any().sum() - shown} more groups)")
        break

### Sanity check on discarded lines

A discarded line should not cover months that the retained line does not.

In [ ]:
chk = []
for gid, g in cl.groupby("group_id"):
    if len(g) < 2:
        continue
    k = g[g["keep"]].iloc[0]
    for _, d in g[~g["keep"]].iterrows():
        before = max(0, (k["first_obs"] - d["first_obs"]).days // 30)
        after = max(0, (d["last_real"] - k["last_real"]).days // 30)
        if before + after > 12:
            chk.append({"group": gid, "kept": k["Symbol"], "dropped": d["Symbol"],
                        "months_before": before, "months_after": after})

chk = pd.DataFrame(chk)
print(f"Discarded lines extending more than 12 months beyond the retained one: {len(chk):,}")
if len(chk):
    print(chk.nlargest(10, ["months_before", "months_after"]).to_string(index=False))
    print("\nInspect these before applying; a large gap may indicate a re-listing "
          "that the overlap threshold has merged.")

## Apply

In [ ]:
monthly_d = monthly[~monthly["symbol"].isin(drop_syms)].copy()
weekly_d  = weekly[~weekly["symbol"].isin(drop_syms)].copy()

print("Monthly:")
print(f"  rows      : {len(monthly):>12,} -> {len(monthly_d):>12,}")
print(f"  securities: {monthly['symbol'].nunique():>12,} -> {monthly_d['symbol'].nunique():>12,}")
print("Weekly:")
print(f"  rows      : {len(weekly):>12,} -> {len(weekly_d):>12,}")
print(f"  securities: {weekly['symbol'].nunique():>12,} -> {weekly_d['symbol'].nunique():>12,}")

## Save

In [ ]:
monthly_d.to_parquet(os.path.join(DATA_DIR, "dedup_monthly.parquet"), index=False)
weekly_d.to_parquet(os.path.join(DATA_DIR, "dedup_weekly.parquet"), index=False)

log_cols = ["country", "full_name", "Symbol", "Activity", "Sector",
            "first_obs", "last_real", "n_months", "avg_mv", "cluster", "keep"]
cl[log_cols].to_csv(os.path.join(DATA_DIR, "deduplication_log.csv"), index=False)

for f in ["dedup_monthly.parquet", "dedup_weekly.parquet"]:
    p = os.path.join(DATA_DIR, f)
    print(f"  {f:28s} {len(pd.read_parquet(p)):>12,} rows  ({os.path.getsize(p)/1e6:.1f} MB)")
print(f"  {'deduplication_log.csv':28s} {len(cl):>12,} rows")
print("\nAnnual data are filtered at panel construction using the retained symbols.")